# 01 - Container volume data: cleaning & exploration

Explore daily container volumes (TEU) handled per commodity by a Port Klang 3PL, check data
quality and build model features. The bundled data (`data/raw/container_volumes.csv`) is
synthetic; any CSV with `date`, a commodity/series column and a TEU/volume column can be used.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src.forecasting.data import FEATURES, HOLIDAYS, build_features, load_volumes

raw = pd.read_csv(ROOT / "data/raw/container_volumes.csv")
raw.head()

## Data quality checks

In [ ]:
print("rows:", len(raw))
print("missing values:\n", raw.isna().sum())
print("negative volumes:", (raw["teu"] < 0).sum())
print("duplicate (commodity, date):", raw.duplicated(["commodity", "date"]).sum())

`load_volumes` normalises commodity names, aggregates duplicates, fills missing days with 0 and clips negatives.

In [ ]:
volumes = load_volumes(ROOT / "data/raw/container_volumes.csv")
volumes.groupby("commodity")["teu"].describe()

## Seasonality
Weekday pattern, palm-oil harvest peak (Sep-Nov) and holiday dips around Chinese New Year and Hari Raya.

In [ ]:
weekday = volumes.assign(dow=volumes["date"].dt.day_name()).groupby("dow")["teu"].mean()
monthly = volumes.assign(month=volumes["date"].dt.month).pivot_table(
    index="month", columns="commodity", values="teu", aggfunc="mean"
)
print(weekday.sort_values(ascending=False).round(1))
monthly.round(1)

In [ ]:
daily = volumes.groupby("date")["teu"].sum()
for name, dates in HOLIDAYS.items():
    for d in dates:
        d = pd.Timestamp(d)
        if d in daily.index:
            around = daily.loc[d - pd.Timedelta(days=3) : d + pd.Timedelta(days=3)].mean()
            month = daily.loc[d - pd.Timedelta(days=30) : d - pd.Timedelta(days=10)].mean()
            print(f"{name:22} {d.date()}  {around:5.1f} TEU/day vs {month:5.1f} normal")

## Feature engineering
Lags (1, 7, 14, 28 days), rolling means/std of *past* values only (no leakage), calendar features and days to the nearest major holiday.

In [ ]:
features = build_features(volumes, sorted(volumes["commodity"].unique()))
features[["date", "commodity", "teu", *FEATURES]].head()

In [ ]:
out = ROOT / "data/processed/features.csv"
features.to_csv(out, index=False)
print(f"saved {len(features):,} rows -> {out}")